<a href="https://colab.research.google.com/github/althafceo/ITA0616-ML/blob/main/CE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import csv
from copy import deepcopy


def load_data(filename):
    with open(filename, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = list(reader)
    return header, rows


def covers(h, x):
    """Check whether hypothesis h covers instance x."""
    return all(hv == "?" or hv == xv for hv, xv in zip(h, x))


def more_general_or_equal(h1, h2):
    """Check whether h1 is more general than or equal to h2."""
    return all(
        a == "?" or a == b or b == "0"
        for a, b in zip(h1, h2)
    )


def minimal_generalizations(h, x):
    """Minimally generalize h to cover positive example x."""
    result = h.copy()

    for i in range(len(h)):
        if result[i] == "0":
            result[i] = x[i]
        elif result[i] != x[i]:
            result[i] = "?"

    return result


def minimal_specializations(g, x, domains, S):
    """Generate minimal specializations of g excluding negative x."""
    results = []

    for i in range(len(g)):
        if g[i] == "?":
            for value in domains[i]:
                if value != x[i]:
                    candidate = g.copy()
                    candidate[i] = value

                    if more_general_or_equal(candidate, S):
                        results.append(candidate)

    return results


def remove_more_specific(hypotheses):
    """Keep only maximally general hypotheses and remove duplicates."""
    unique = []

    for h in hypotheses:
        if h not in unique:
            unique.append(h)

    return [
        h for h in unique
        if not any(
            h != other and more_general_or_equal(other, h)
            for other in unique
        )
    ]


def remove_more_general(hypotheses):
    """Keep only maximally specific hypotheses and remove duplicates."""
    unique = []

    for h in hypotheses:
        if h not in unique:
            unique.append(h)

    return [
        h for h in unique
        if not any(
            h != other and more_general_or_equal(h, other)
            for other in unique
        )
    ]


def candidate_elimination(rows, n_attrs):
    domains = [
        sorted({row[i] for row in rows})
        for i in range(n_attrs)
    ]

    S = [["0"] * n_attrs]
    G = [["?"] * n_attrs]

    for idx, row in enumerate(rows, start=1):
        x = row[:-1]
        label = row[-1].strip().lower()

        if label == "yes":
            # Remove hypotheses from G that do not cover x.
            G = [g for g in G if covers(g, x)]

            # Generalize S to cover x.
            S = [
                minimal_generalizations(s, x)
                if not covers(s, x) else s
                for s in S
            ]

            # Keep S hypotheses that are no more general than a G hypothesis.
            S = [
                s for s in S
                if any(more_general_or_equal(g, s) for g in G)
            ]
            S = remove_more_general(S)

        elif label == "no":
            # Remove hypotheses from S that cover x.
            S = [s for s in S if not covers(s, x)]

            # Specialize G hypotheses that cover x.
            new_G = []

            for g in G:
                if covers(g, x):
                    candidates = minimal_specializations(
                        g, x, domains, S[0] if S else ["0"] * n_attrs
                    )
                    new_G.extend(candidates)
                else:
                    new_G.append(g)

            G = [
                g for g in new_G
                if any(more_general_or_equal(g, s) for s in S)
            ]
            G = remove_more_specific(G)

        else:
            raise ValueError(f"Unknown class label: {row[-1]}")

        print(f"Example {idx}: {x} -> {label.title()}")
        print(f"   S = {S}")
        print(f"   G = {G}\n")

    return S, G


if __name__ == "__main__":
    header, data = load_data("/content/Candidate.csv")

    print("Attributes:", header[:-1], "\n")

    S, G = candidate_elimination(data, len(header) - 1)

    print("Final Specific boundary S:", S)
    print("Final General boundary G :", G)

Attributes: ['Size', 'Color', 'Shape'] 

Example 1: ['Big', 'Red', 'Circle'] -> No
   S = [['0', '0', '0']]
   G = [['Small', '?', '?'], ['?', 'Blue', '?'], ['?', '?', 'Triangle']]

Example 2: ['Small', 'Red', 'Triangle'] -> No
   S = [['0', '0', '0']]
   G = [['Small', '?', 'Circle'], ['?', 'Blue', '?'], ['Big', '?', 'Triangle']]

Example 3: ['Small', 'Red', 'Circle'] -> Yes
   S = [['Small', 'Red', 'Circle']]
   G = [['Small', '?', 'Circle']]

Example 4: ['Big', 'Blue', 'Circle'] -> No
   S = [['Small', 'Red', 'Circle']]
   G = [['Small', '?', 'Circle']]

Example 5: ['Small', 'Blue', 'Circle'] -> Yes
   S = [['Small', '?', 'Circle']]
   G = [['Small', '?', 'Circle']]

Final Specific boundary S: [['Small', '?', 'Circle']]
Final General boundary G : [['Small', '?', 'Circle']]
